<a href="https://colab.research.google.com/github/dnhoang-2006/introds-arxiv-project/blob/feat%2Fcolab-benchmark/book/milestone2/colab_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Đo lường hiệu năng bộ bóc tách trên Google Colab CPU-only

> **Môn học:** Nhập môn Khoa học Dữ liệu - HCMUS
> **Milestone 2:** Bóc tách Cấu trúc Tài liệu & Chuẩn Hóa Phân Cấp
> **Thành viên phụ trách:** Dương Ngọc Minh Thư (Đo hiệu năng & Song song hoá `--workers`)
> **Mục tiêu:** Đo đúng bốn chỉ số hiệu năng đề bài yêu cầu cho `src/parse_main.py` trên testbed chuẩn Google Colab CPU-only, theo cách kiểm chứng được, và so sánh chế độ tuần tự với song song.

---

### Yêu cầu của đề bài

Đề bài quy định: *running time* gồm **thời gian end-to-end của parser** và **thời gian trung bình xử lý một publication**; *memory footprint* gồm **RAM tối đa** và **dung lượng output cuối cùng**. Testbed chuẩn là **Google Colab ở chế độ CPU-only**.

| Chỉ số | Nguồn trong `logs/benchmark_runs.csv` |
|---|---|
| Wall-clock toàn bộ parser | `wall_time_seconds` |
| Thời gian trung bình mỗi publication | `wall_time_seconds / N` |
| RAM tối đa | `peak_ram_mb` (kèm `peak_ram_source`) |
| Dung lượng output cuối | `disk_usage_mb`, đo trên thư mục `parsed/` |

### Nguyên tắc đo kiểm chứng được

- Mọi con số do `src/benchmark.py` tự ghi; `run_id` có hash ngẫu nhiên, **không nhập tay**.
- Wall-clock bao gồm cả khởi động tiến trình và I/O, nên với `--workers 1` nó **luôn lớn hơn** tổng cột `seconds` của `logs/parse.csv`. Notebook tự kiểm tra điều kiện này.
- Trước mỗi lần đo, `parsed/` và `logs/parse.csv` được xoá, vì `parse_main.py` bỏ qua publication đã parse — không xoá thì lần đo sau sai và `disk_usage_mb` ≈ 0.

### Cách chạy

1. `Runtime → Change runtime type → CPU`.
2. Chuẩn bị `corpus_sample.zip` (N publication đầu của file ID) trên máy cá nhân, vì corpus không nằm trong repo.
3. Chạy lần lượt từ trên xuống; đến mục 3 thì chọn file zip.

> ⚠️ Notebook này **không** nằm trong `_toc.yml`: nó cần upload file thủ công và corpus thật, trong khi book build với `execute_notebooks: force`. Kết quả được xuất sang `book/data/` để chương báo cáo hiệu năng đọc lại.

## 0. Cấu hình thí nghiệm và hàm tiện ích

In [1]:
from __future__ import annotations

import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import pandas as pd
from google.colab import files

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

# --- Thông số thí nghiệm: chỉ sửa ở đây ---
IDS_FILE = "ids/24120144.txt"  # file ID của thành viên
N_PUBS = 50                    # cỡ mẫu, phải khớp số publication trong corpus_sample.zip
SMOKE_N_PUBS = 5               # cỡ mẫu chạy thử trước khi đo thật
BRANCH = "main"                # nhánh mã nguồn được đo
RUN_PARALLEL = True            # False: chỉ chạy phép đo bắt buộc (--workers 1)

# --- Đường dẫn ---
REPO_URL = "https://github.com/dnhoang-2006/introds-arxiv-project.git"
REPO_DIR = Path("/content/introds-arxiv-project")
CORPUS_DIR = REPO_DIR / "corpus"
PARSED_DIR = REPO_DIR / "parsed"
LOGS_DIR = REPO_DIR / "logs"
BENCHMARK_CSV = LOGS_DIR / "benchmark_runs.csv"
PARSE_LOG = LOGS_DIR / "parse.csv"
EXPORT_DIR = REPO_DIR / "export"
EXPORT_ZIP = REPO_DIR / "thu_m2_benchmark.zip"


def run_command(*cmd: str, cwd: Path = REPO_DIR, echo: bool = True) -> str:
    """Chạy một lệnh, in output (nếu echo) và trả stdout. Lệnh lỗi thì ném CalledProcessError."""
    proc = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if echo and (proc.stdout or proc.stderr):
        print((proc.stdout + proc.stderr).rstrip())
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd, proc.stdout, proc.stderr)
    return proc.stdout.strip()


def parse_log_copy(workers: int, n_pubs: int) -> Path:
    """Bản sao parse.csv của một lần đo, để đối chiếu sau khi lần đo kế tiếp xoá log."""
    return LOGS_DIR / f"parse_w{workers}_n{n_pubs}.csv"

## 1. Chuẩn bị mã nguồn và môi trường

In [2]:
if not REPO_DIR.exists():
    run_command("git", "clone", "-q", REPO_URL, str(REPO_DIR), cwd=REPO_DIR.parent)
os.chdir(REPO_DIR)

run_command("git", "checkout", "-q", "--", ".")  # bỏ mọi sửa đổi tạm trên Colab
run_command("git", "fetch", "-q", "origin")
run_command("git", "checkout", "-q", BRANCH)
run_command("git", "pull", "-q")
COMMIT = run_command("git", "rev-parse", "--short", "HEAD", echo=False)
run_command("git", "log", "-1", "--date=iso", "--format=Mã nguồn: commit %h — %ad — %s")
run_command(sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt")

Mã nguồn: commit eb517af — 2026-11-20 16:53:10 +0700 — Merge pull request #36 from dnhoang-2006/feat/bib


''

## 2. Thông tin testbed

In [3]:
def read_cpu_model() -> str:
    """Tên CPU đọc từ /proc/cpuinfo."""
    match = re.search(r"model name\s*:\s*(.+)", Path("/proc/cpuinfo").read_text())
    return match.group(1).strip() if match else "không rõ"


def read_total_ram_gb() -> float:
    """Tổng RAM của máy, đơn vị GB."""
    match = re.search(r"MemTotal:\s+(\d+)", Path("/proc/meminfo").read_text())
    return int(match.group(1)) / 1024**2


has_gpu = shutil.which("nvidia-smi") is not None
print(f"Python : {sys.version.split()[0]}")
print(f"CPU    : {read_cpu_model()} × {os.cpu_count()} vCPU")
print(f"RAM    : {read_total_ram_gb():.1f} GB")
print(f"GPU    : {'CÓ — đổi sang runtime CPU rồi chạy lại' if has_gpu else 'không — đúng chế độ CPU-only'}")

Python : 3.13.18
CPU    : Intel(R) Xeon(R) CPU @ 2.20GHz × 2 vCPU
RAM    : 12.7 GB
GPU    : không — đúng chế độ CPU-only


## 3. Nạp mẫu corpus

In [4]:
if not CORPUS_DIR.exists():
    uploaded = files.upload()  # chọn corpus_sample.zip
    with zipfile.ZipFile(next(iter(uploaded))) as archive:
        archive.extractall(REPO_DIR)

sys.path.insert(0, str(REPO_DIR / "src"))
import config  # noqa: E402 — module của repo, chỉ import được sau khi có src/ trong sys.path

# parse_main.py lặng lẽ bỏ qua ID không có trong corpus, nên phải kiểm trước khi đo.
sample_ids = list(dict.fromkeys(config.strip_version(i) for i in config.load_ids(IDS_FILE)))[:N_PUBS]
missing_ids = [i for i in sample_ids if not config.pub_dir(i).exists()]
print(f"Corpus có {len(sample_ids) - len(missing_ids)}/{len(sample_ids)} publication đầu của {IDS_FILE}")
if missing_ids:
    raise RuntimeError(f"Thiếu {len(missing_ids)} publication trong corpus, ví dụ: {missing_ids[:5]}")

Saving corpus_sample.zip to corpus_sample.zip
Corpus có 50/50 publication đầu của ids/24120144.txt


## 4. Hàm đo

`benchmark.py` là wrapper: lệnh cần đo đặt sau dấu `--`. Mỗi lần đo lưu riêng bản sao `parse.csv` để đối chiếu sau.

In [5]:
def run_benchmark(workers: int, n_pubs: int) -> None:
    """Đo một lần parse_main.py qua benchmark.py, ghi một dòng vào BENCHMARK_CSV.

    Xoá parsed/ và parse.csv trước, vì parse_main.py bỏ qua publication đã parse.
    """
    shutil.rmtree(PARSED_DIR, ignore_errors=True)
    PARSE_LOG.unlink(missing_ok=True)
    notes = f"Colab CPU-only, {n_pubs} publications, workers={workers}, commit {COMMIT}"
    run_command(
        sys.executable, "src/benchmark.py",
        "--stage", "parse",
        "--disk-path", str(PARSED_DIR),
        "--csv", str(BENCHMARK_CSV),
        "--notes", notes,
        "--",
        sys.executable, "src/parse_main.py",
        "--ids", IDS_FILE,
        "--limit", str(n_pubs),
        "--workers", str(workers),
        "--quiet",
    )
    shutil.copy(PARSE_LOG, parse_log_copy(workers, n_pubs))

## 5. Chạy thử trên mẫu nhỏ

Xác nhận môi trường trước khi đo thật. Ô này báo lỗi thì dừng lại xem lỗi.

In [6]:
run_benchmark(workers=1, n_pubs=SMOKE_N_PUBS)


Publication     : 5
Version theo status: ok=5, empty=1, fail=0, skipped=0
Thời gian       : 1.1s
[benchmark] Logged to CSV: /content/introds-arxiv-project/logs/benchmark_runs.csv

BENCHMARK REPORT — run_20261120_104812_7b670fe8
Command          : /usr/bin/python3 src/parse_main.py --ids ids/24120144.txt --limit 5 --workers 1 --quiet
Stage            : parse
Paper ID         : N/A
Status           : SUCCESS (Exit code: 0)
Wall-clock Time  : 1.3156 s
Peak RAM         : 27.57 MB (sampling)
Average RAM      : 24.18 MB
RAM Interval     : 0.05 s
Disk Usage       : +0.9042 MB (Scope: delta:parsed)
Timestamps (UTC) : 2026-11-20T10:48:12.146310+00:00 -> 2026-11-20T10:48:13.461917+00:00
Testbed          : Linux-6.6.159+-x86_64 (Python 3.13.18, RAM 12.7GB)
Notes            : Colab CPU-only, 5 publications, workers=1, commit eb517af

File BibTeX /content/introds-arxiv-project/corpus/2505-00667/v1/Bib.bib chứa 2 block lỗi cú pháp bị bỏ qua (lenient mode)


## 6. Phép đo bắt buộc — tuần tự (`--workers 1`)

In [7]:
run_benchmark(workers=1, n_pubs=N_PUBS)


Publication     : 50
Version theo status: ok=84, empty=2, fail=0, skipped=0
Thời gian       : 27.3s
[benchmark] Logged to CSV: /content/introds-arxiv-project/logs/benchmark_runs.csv

BENCHMARK REPORT — run_20261120_104813_40e73cb3
Command          : /usr/bin/python3 src/parse_main.py --ids ids/24120144.txt --limit 50 --workers 1 --quiet
Stage            : parse
Paper ID         : N/A
Status           : SUCCESS (Exit code: 0)
Wall-clock Time  : 27.5223 s
Peak RAM         : 35.69 MB (sampling)
Average RAM      : 30.82 MB
RAM Interval     : 0.05 s
Disk Usage       : +13.9157 MB (Scope: delta:parsed)
Timestamps (UTC) : 2026-11-20T10:48:13.683131+00:00 -> 2026-11-20T10:48:41.205477+00:00
Testbed          : Linux-6.6.159+-x86_64 (Python 3.13.18, RAM 12.7GB)
Notes            : Colab CPU-only, 50 publications, workers=1, commit eb517af

File BibTeX /content/introds-arxiv-project/corpus/2505-00667/v1/Bib.bib chứa 2 block lỗi cú pháp bị bỏ qua (lenient mode)
File BibTeX /content/introds-arxiv-p

## 7. Phép đo mở rộng — song song (`--workers 2`)

Colab miễn phí có 2 vCPU nên `--workers 2` là mức song song có ý nghĩa. Phần này là điểm cộng; lỗi thì bỏ qua, không ảnh hưởng bốn chỉ số bắt buộc.

In [8]:
if RUN_PARALLEL:
    run_benchmark(workers=2, n_pubs=N_PUBS)


Publication     : 50
Version theo status: ok=84, empty=2, fail=0, skipped=0
Thời gian       : 22.8s
[benchmark] Logged to CSV: /content/introds-arxiv-project/logs/benchmark_runs.csv

BENCHMARK REPORT — run_20261120_104841_2e1b663b
Command          : /usr/bin/python3 src/parse_main.py --ids ids/24120144.txt --limit 50 --workers 2 --quiet
Stage            : parse
Paper ID         : N/A
Status           : SUCCESS (Exit code: 0)
Wall-clock Time  : 23.0096 s
Peak RAM         : 84.59 MB (sampling)
Average RAM      : 79.01 MB
RAM Interval     : 0.05 s
Disk Usage       : +13.9157 MB (Scope: delta:parsed)
Timestamps (UTC) : 2026-11-20T10:48:41.430812+00:00 -> 2026-11-20T10:49:04.440434+00:00
Testbed          : Linux-6.6.159+-x86_64 (Python 3.13.18, RAM 12.7GB)
Notes            : Colab CPU-only, 50 publications, workers=2, commit eb517af

File BibTeX /content/introds-arxiv-project/corpus/2505-00667/v1/Bib.bib chứa 2 block lỗi cú pháp bị bỏ qua (lenient mode)
File BibTeX /content/introds-arxiv-p

## 8. Kết quả đo

In [9]:
def load_runs(n_pubs: int, commit: str) -> pd.DataFrame:
    """Các lần đo với đúng cỡ mẫu và commit, mỗi mức workers lấy lần mới nhất."""
    runs = pd.read_csv(BENCHMARK_CSV)
    tag = runs["notes"].fillna("")
    runs = runs[tag.str.contains(f", {n_pubs} publications,", regex=False)
                & tag.str.endswith(f"commit {commit}")].copy()
    runs["workers"] = runs["notes"].str.extract(r"workers=(\d+)")[0].astype(int)
    runs["avg_seconds_per_pub"] = runs["wall_time_seconds"] / n_pubs
    return runs.drop_duplicates("workers", keep="last").set_index("workers").sort_index()


runs = load_runs(N_PUBS, COMMIT)
runs[["run_id", "status", "exit_code", "testbed", "wall_time_seconds", "avg_seconds_per_pub",
      "peak_ram_mb", "peak_ram_source", "average_ram_mb", "disk_usage_mb"]]

,run_id,status,exit_code,testbed,wall_time_seconds,avg_seconds_per_pub,peak_ram_mb,peak_ram_source,average_ram_mb,disk_usage_mb
workers,,,,,,,,,,
1,run_20261120_104813_40e73cb3,success,0,"Linux-6.6.159+-x86_64 (Python 3.13.18, RAM 12.7GB)",27.5223,0.550446,35.69,sampling,30.82,13.9157
2,run_20261120_104841_2e1b663b,success,0,"Linux-6.6.159+-x86_64 (Python 3.13.18, RAM 12.7GB)",23.0096,0.460192,84.59,sampling,79.01,13.9157


### 8.1. Kiểm định tính hợp lệ

`peak_ram_source`: công cụ lấy giá trị **lớn hơn** giữa `getrusage` (kernel, bắt được đỉnh nhọn giữa hai lần lấy mẫu) và `sampling` (psutil, cộng dồn cả cây tiến trình). Với `--workers 2`, RAM là tổng của nhiều tiến trình con nên thường ra `sampling` — đây là hành vi đúng.

Với `--workers 1`, wall-clock phải lớn hơn tổng `seconds`. Với `--workers 2`, tổng `seconds` có thể vượt wall-clock vì các publication được xử lý chồng thời gian.

In [10]:
for workers, run in runs.iterrows():
    parse_log = pd.read_csv(parse_log_copy(workers, N_PUBS))
    n_parsed = parse_log["arxiv_id"].nunique()
    if n_parsed != N_PUBS:
        print(f"[LỖI] workers={workers}: parse.csv chỉ có {n_parsed}/{N_PUBS} publication")
    item_seconds = parse_log["seconds"].sum()
    wall_seconds = run["wall_time_seconds"]
    print(f"workers={workers}: wall-clock = {wall_seconds:.2f}s | tổng seconds = {item_seconds:.2f}s | "
          f"trạng thái version: {parse_log['status'].value_counts().to_dict()}")
    if workers == 1:
        verdict = "[OK] wall-clock > tổng seconds" if wall_seconds > item_seconds \
            else "[LỖI] wall-clock <= tổng seconds, kiểm tra lại"
        print(f"   {verdict}")

workers=1: wall-clock = 27.52s | tổng seconds = 17.37s | trạng thái version: {'ok': 84, 'empty': 2}
   [OK] wall-clock > tổng seconds
workers=2: wall-clock = 23.01s | tổng seconds = 25.73s | trạng thái version: {'ok': 84, 'empty': 2}


### 8.2. Bảng tổng hợp bốn chỉ số

In [11]:
METRICS = {
    "Wall-clock toàn bộ parser (s)": "wall_time_seconds",
    "Trung bình mỗi publication (s)": "avg_seconds_per_pub",
    "RAM tối đa (MB)": "peak_ram_mb",
    "Dung lượng output cuối (MB)": "disk_usage_mb",
}

summary = pd.DataFrame(
    {f"--workers {w}": [runs.loc[w, col] for col in METRICS.values()] for w in runs.index},
    index=list(METRICS),
).round(3)
if {1, 2} <= set(runs.index):
    baseline = summary["--workers 1"].where(summary["--workers 1"] != 0)
    change = summary["--workers 2"] / baseline - 1
    summary["Thay đổi"] = change.map(lambda x: "—" if pd.isna(x) else f"{x:+.1%}")

print(f"Testbed: Google Colab CPU-only · Mẫu: {N_PUBS} publication ({IDS_FILE}) · Commit: {COMMIT}\n")
print(summary.to_markdown())

Testbed: Google Colab CPU-only · Mẫu: 50 publication (ids/24120144.txt) · Commit: eb517af

|                                |   --workers 1 |   --workers 2 | Thay đổi   |
|:-------------------------------|--------------:|--------------:|:-----------|
| Wall-clock toàn bộ parser (s)  |        27.522 |        23.01  | -16.4%     |
| Trung bình mỗi publication (s) |         0.55  |         0.46  | -16.4%     |
| RAM tối đa (MB)                |        35.69  |        84.59  | +137.0%    |
| Dung lượng output cuối (MB)    |        13.916 |        13.916 | +0.0%      |


## 9. Xuất kết quả

Các tệp đặt tên theo quy ước `thu_*` của `book/data/`. Giải nén và chép vào `book/data/` để chương báo cáo hiệu năng Milestone 2 đọc lại khi build book.

In [12]:
EXPORT_DIR.mkdir(exist_ok=True)
runs.reset_index().to_csv(EXPORT_DIR / "thu_m2_benchmark_runs.csv", index=False)
shutil.copy(BENCHMARK_CSV, EXPORT_DIR / "thu_m2_benchmark_runs_raw.csv")  # bản gốc đầy đủ
for workers in runs.index:
    shutil.copy(parse_log_copy(workers, N_PUBS), EXPORT_DIR / f"thu_m2_parse_w{workers}.csv")

with zipfile.ZipFile(EXPORT_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EXPORT_DIR.glob("*.csv")):
        archive.write(path, path.name)
        print(f"  {path.name}")
files.download(str(EXPORT_ZIP))

  thu_m2_benchmark_runs.csv
  thu_m2_benchmark_runs_raw.csv
  thu_m2_parse_w1.csv
  thu_m2_parse_w2.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>